# Transmon-controlled cavity–cavity SWAP via a **single-cavity controlled parity**

Hardware assumed (all gates **ideal**):

* cavity–cavity beamsplitter (good, and used as a primitive here)
* transmon–cavity **ge** exchange, **one cavity at a time, one pump tone**
* transmon g–e used as the control qubit; the $f$ level is never used
* no static $\chi$ — the transmon is idle unless a pump is on

**Figure of merit: total $g\,T$**, where $g$ is the transmon–cavity exchange rate.

---

## The decomposition

$$\mathrm{SWAP}_{12}=e^{\,i\pi n_-},\qquad a_-=\frac{a_1-a_2}{\sqrt2}$$

and a single 50:50 beamsplitter $V$ satisfies $V^\dagger n_1 V = n_-$, so

$$\boxed{\;\text{c-SWAP}\;=\;V^\dagger\cdot\big(\text{transmon-controlled parity of cavity 1}\big)\cdot V\;}$$

Everything hard is in the middle gate, which acts on **one** cavity. Only one pump
tone is ever on, which avoids the two-tone heating of the coupler that the
simultaneous-drive variant would incur.

The middle gate is

$$U_{\rm target}=|g\rangle\langle g|\otimes\mathbb{1}\;+\;|e\rangle\langle e|\otimes e^{i\pi \hat n}$$

## Primary drive

**One** pump, activating the ge exchange between the transmon and cavity 1:

$$H_j=\underbrace{\Delta_j|e\rangle\langle e|}_{\text{sideband detuning}}
\;+\;g\left(e^{i\phi_j}\,a^\dagger|g\rangle\langle e|+\mathrm{h.c.}\right),
\qquad\text{for a time }t_j$$

Segment $j$ is three numbers: **pulse area** $x_j=g\,t_j$, **drive phase** $\phi_j$,
**detuning** $r_j=\Delta_j/g$. Only one cavity is driven, so there is no supermode
$\sqrt2$ enhancement and

$$g\,T_{\rm total}=\sum_j x_j$$

## Control qubit

The **ordinary transmon g–e qubit**; parity is applied when the transmon is in
$|e\rangle$. Both control states sit in the *same* driven block
$\{|m,g\rangle,|m\!-\!1,e\rangle\}$, which is what makes the design easy:

* the ge exchange conserves $n+n_q$, so the propagator is block diagonal in $2\times2$ blocks;
* requiring $|\langle m,g|U|m,g\rangle|=1$ **forces every block diagonal** — no residual
  g–e mixing (approached to within the fidelity budget);
* unitarity then fixes the $e$ phase from the $g$ phase,
  $U^{gg}_m=e^{i(\gamma+\psi_m)}$, $U^{ee}_m=e^{i(\gamma-\psi_m)}$ with
  $\gamma=-\tfrac12\sum_j\Delta_jt_j$ independent of $m$.

## The target profile

Controlled parity wants $\langle n,e|U|n,e\rangle/\langle n,g|U|n,g\rangle=e^{i(\pi n+c)}$.
Those two amplitudes come from **adjacent** blocks ($m=n$ and $m=n+1$), so the
condition is a recurrence rather than a direct phase assignment:

$$\psi_{n+1}+\psi_n=-\pi n+c'\qquad\Longrightarrow\qquad \psi_m=-\frac{\pi}{2}m$$

(the free constant $c'=-\pi/2$ linearises it). So the design target is the
"quarter parity"

$$\langle m,g|U|m,g\rangle=e^{-i\pi m/2}$$

and the leftover unconditional $e^{-i\pi n/2}$ is a plain **cavity virtual Z — free**.
No beamsplitter is needed to finish the parity itself.

## Fock range

To cover cavity Fock $0..n_{\rm cut}$ on **both** branches we need the $g$ branch for
$m=0..n_{\rm cut}$ and the $e$ branch for $m=1..n_{\rm cut}+1$, i.e. blocks
$m=0..n_{\rm cut}+1$. For $n_{\rm cut}=8$ that is $n_{\max}=9$.

> $V$ mixes the cavities, so cavity 1 after $V$ holds up to $n_1+n_2$ photons.
> A parity exact to Fock 8 therefore makes the c-SWAP exact exactly when the
> **total** photon number is $\le 8$ — verified explicitly in section 4.

## Results

Fidelity floor 0.9999 on the parity gate over cavity Fock 0–8:

| sequence | $M$ | detuning | $g\,T$ | $F_{\rm parity}$ (Fock 0–8) |
|---|---|---|---|---|
| `cparity_fock8.npz` | 10 | on | **11.58** | 0.9999090 |
| `cparity_fock8_resonant.npz` | 16 | off | 20.70 | 1.0000000 |

Allowing sideband detuning buys ~42%. The resonant one runs in
`optimal_control.py` unmodified (its `"bs"` gate has no detuning knob).

The $g\,T=12.06$ point has the fidelity budget **spent**: six parallel fixed-area descents
(different seeds, and padded to $M=24,28,32,40$) all failed to get below it. So this is
a real wall for this ansatz at this floor, not merely where the search was stopped.
Residual g–e block mixing is $\sim7\times10^{-3}$, which is exactly the
$\sim6\times10^{-5}$ infidelity.

### Gate count is separately reducible

$g\,T$ and $M$ trade off only weakly. Holding the area fixed at 12.0616 and chaining $M$
downward (deleting the smallest-area segment and re-optimising at fixed area) shrinks the
sequence at no cost at all, and the freed fidelity budget then lets $g\,T$ drop again.
Alternating the two stages went $g\,T=12.06$ at $M=20$ → **11.58 at $M=10$**:

| $M$ | $F_{\rm parity}$ (Fock 0–8) at $g\,T=12.0616$ |
|---|---|
| 20 | 0.9999130 |
| 16 | 0.9999155 |
| 14 | 0.9999142 |
| 13 | 0.9999528 |
| 12 | 0.9999433 |
| 11 | 0.9999684 |
| **11** (after a second $g\,T$ descent to 11.58) | 0.9999284 |
| **10** (at $g\,T=11.58$) | **0.9999090** |

So the $M=20$ sequence was badly padded: it contained two segments of area $\sim10^{-3}$
(pure no-ops) plus several more below 1% of the total. Fewer, larger segments describe the
same gate better — at $M=11$ the residual g–e block mixing drops to $3.7\times10^{-3}$
from $7.5\times10^{-3}$ at $M=12$.

Parameter counting allows $M\ge6$: at $M=11$ there are $3M-1=32$ free knobs against
$2n_{\max}-1=17$ conditions. Whether $M<11$ is reachable was not settled — the chained
search was still running when it was stopped, and different random seeds broke through at
different $M$ (one stalled at 14, two at 12, one reached 11), so the wall is the
optimiser's, not the counting bound's.

## 0. Notation — every symbol and code variable

**Indices and ranges**

| symbol | code | meaning |
|---|---|---|
| $n$ | `n`, `nn` | cavity Fock (photon) number |
| $n_{\rm cut}$ | `NCUT`, `ncut` | highest cavity Fock level the gate must be correct on — **8** here |
| $n_q$ | — | transmon excitation, 0 for $\lvert g\rangle$ and 1 for $\lvert e\rangle$ |
| $m$ | `m` | block label = total excitation $n+n_q$; the ge exchange conserves it |
| $n_{\max}$ | `NMAX`, `nmax` | highest block needed, $=n_{\rm cut}+1=\mathbf{9}$ |
| $M$ | **`M`** | **number of pulse segments in the sequence** (e.g. 20) |
| $j$ | `j` | segment index, $1\ldots M$ |

**Pulse parameters** — the sequence is stored as one flat vector
`params` (also called `p`) $=[\,x_0\ldots x_{M-1},\ \phi_0\ldots\phi_{M-1},\ r_0\ldots r_{M-1}\,]$
(the $r$ block is absent for a resonant sequence):

| symbol | code | meaning |
|---|---|---|
| $g$ | `g` | transmon–cavity ge exchange rate — the hardware-limited rate |
| $t_j$ | — | duration of segment $j$ |
| $x_j$ | `x` $=$ `params[:M]` | **pulse area** $x_j=g\,t_j$, dimensionless (= `rotation` in `optimal_control`) |
| $\phi_j$ | `phi` $=$ `params[M:2M]` | drive phase of segment $j$ (= `phase` in `optimal_control`) |
| $\Delta_j$ | — | sideband detuning of segment $j$ |
| $r_j$ | `r` $=$ `params[2M:3M]` | detuning in units of $g$: $r_j=\Delta_j/g$ |
| — | `USE_DET`, `det` | whether this sequence uses the detuning knob at all |

**Figure of merit**

| symbol | code | meaning |
|---|---|---|
| $T$ | — | total gate time $\sum_j t_j$ |
| $g\,T$ | **`gT`** | **what we minimise**; $g\,T=\sum_j x_j$ since $g\,t_j=x_j$ |
| $F$ | `F_blocks`, `F_dense`, `F_oc` | projected gate fidelity on Fock $0..n_{\rm cut}\times\{g,e\}$ |
| — | `cp.FLOOR_DEFAULT` | fidelity floor the optimiser must respect (0.9999) |

**Design quantities**

| symbol | code | meaning |
|---|---|---|
| $\theta$ | `THETA`, `theta` | target slope of the g-branch phase profile; $\theta=-\pi/2$ |
| $\psi_m$ | — | g-branch phase in block $m$; the target is $\psi_m=\theta\,m$ |
| $\gamma$ | — | $-\tfrac12\sum_j\Delta_j t_j$; a common offset, the same for every block |
| $u_m$ | `u` | $\langle m,g\rvert U\lvert m,g\rangle$ — the g-branch diagonal element |
| $v_m$ | `v` | $\langle m\!-\!1,e\rvert U\lvert m\!-\!1,e\rangle$ — the e-branch diagonal element |
| $U^{gg}_m,\;U^{ee}_m$ | — | same two quantities written as block matrix elements |
| $c'$ | `cprime` | free constant in the recurrence; $c'=-\pi/2$ linearises $\psi_m$ |

**The beamsplitter sandwich**

| symbol | code | meaning |
|---|---|---|
| $a_1,a_2$ | — | the two cavity annihilation operators |
| $a_-$ | — | antisymmetric supermode $(a_1-a_2)/\sqrt2$ |
| $n_-$ | — | $a_-^\dagger a_-$; note $\mathrm{SWAP}_{12}=e^{i\pi n_-}$ |
| $V$ | `V` | 50:50 cavity–cavity beamsplitter, chosen so $V^\dagger n_1 V=n_-$ |
| — | `N_c`, `Nc` | cavity Hilbert-space truncation used in simulation (needs $>n_{\rm cut}$) |
| — | `N_q`, `Nq` | transmon levels kept, 2 (only g and e are ever used) |
| — | `SEQ` | filename of the stored sequence being analysed |

In [ ]:
import time
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
import controlled_parity as cp
import controlled_swap as cs

%load_ext autoreload
%autoreload 2
np.set_printoptions(precision=5, suppress=True, linewidth=140)
print("fidelity floor used by the optimiser:", cp.FLOOR_DEFAULT)

## 1. The target profile, checked

Solve the recurrence $\psi_{n+1}+\psi_n=-\pi n+c'$ numerically and confirm (a) that any
$c'$ gives the right ratio up to a constant, and (b) that $c'=-\pi/2$ makes it
$\psi_m=-\pi m/2$.

In [ ]:
NCUT = 8                      # n_cut: gate must be correct on cavity Fock 0..8
NMAX = cp.target_nmax(NCUT)   # n_max: highest block m needed = NCUT + 1 = 9
print(f"ncut = {NCUT}  ->  nmax = {NMAX} blocks")

for cprime in (-np.pi / 2, 0.0, 0.7):
    psi = np.zeros(NMAX + 2)
    for n in range(NMAX + 1):
        psi[n + 1] = -np.pi * n + cprime - psi[n]
    ratio = np.array([np.angle(np.exp(-1j * (psi[n + 1] + psi[n])))
                      for n in range(NMAX + 1)])
    want = np.array([np.angle(np.exp(1j * np.pi * n)) for n in range(NMAX + 1)])
    ok = np.allclose(np.exp(1j * (ratio - want)), np.exp(1j * (ratio[0] - want[0])))
    lin = np.allclose(np.exp(1j * psi), np.exp(-1j * np.pi * np.arange(NMAX + 2) / 2))
    print(f"  c' = {cprime:+.4f}:  ratio = pi*n + const -> {ok};   "
          f"psi_m == -pi*m/2 -> {lin}")

## 2. The minimum-$g\,T$ sequence

`cparity_fock8.npz` is the stored optimum, $g\,T=12.06$. To search again from scratch
use `cp.produce_optimum(ncut=8, ...)`; to try to beat it, warm-start a fixed-area descent
from the stored sequence (optionally padded with extra zero-area segments, which adds
knobs without adding time). Six such descents stalled at 12.06 with the fidelity budget
spent, so expect diminishing returns unless the floor is relaxed further.

In [ ]:
SEQ = "cparity_fock8.npz"        # or "cparity_fock8_resonant.npz"
# p       : flat parameter vector [x_j, phi_j, r_j]
# M       : number of pulse segments
# THETA   : target phase slope (-pi/2)
# NCUT_S  : Fock cutoff this sequence was designed for
# USE_DET : does it use the detuning knob?
p, M, THETA, NCUT_S, USE_DET = cp.load_sequence(SEQ)
gT = float(p[:M].sum())          # g*T -- the figure of merit
print(f"{SEQ}:  M = {M} segments,  detuning {'on' if USE_DET else 'off'},  "
      f"theta = {THETA/np.pi:+.2f} pi,  Fock 0..{NCUT_S}")
print(f"  g*T = {gT:.4f}")
for g_MHz in (0.5, 1.0, 2.0, 3.0):
    print(f"    g/2pi = {g_MHz:4.1f} MHz  ->  T_parity = {gT/(2*np.pi*g_MHz):6.3f} us")

### The pulse table

$x_j=g\,t_j$ (so $t_j=x_j/g$), $\phi_j$ the drive phase, $\Delta_j/g$ the sideband
detuning. These map one-to-one onto `optimal_control`'s `"bs"` gate parameters
(`rotation` $=x_j$, `phase` $=\phi_j$); the detuning column needs a gate with a
$|e\rangle\langle e|$ term added to the registry.

In [ ]:
x = p[:M]                                    # x_j  = g * t_j   (pulse area)
phi = p[M:2 * M]                             # phi_j            (drive phase)
r = p[2 * M:3 * M] if USE_DET else np.zeros(M)   # r_j = Delta_j / g  (detuning)
print(f"{'j':>3} {'x_j = g t_j':>13} {'phi_j [rad]':>14} {'Delta_j / g':>13}")
for j in range(M):
    print(f"{j+1:3d} {x[j]:13.6f} {phi[j]:14.6f} {r[j]:13.6f}")
print(f"{'':3} {gT:13.6f}   <- g*T")

## 3. Verification of the parity gate

Three independent routes:

1. **fast metric** — projected gate fidelity computed from the $2\times2$ blocks;
2. **dense propagation** — build every $H_j$ and `expm` it in the full
   cavity $\otimes$ transmon space, using no block reduction at all;
3. **`optimal_control.propagate_gates`** — rebuild the same propagator with the
   project's own gate registry (resonant sequences only, since the built-in `"bs"`
   has no detuning parameter).

Routes 1 and 2 must agree to machine precision, and route 3 must reproduce the
unitary itself to machine precision.

In [ ]:
F_blocks = cp.gate_fidelity_blocks(p, M, ncut=NCUT, use_detuning=USE_DET)
F_dense = cp.cparity_fidelity(p, M, ncut=NCUT, use_detuning=USE_DET)
print(f"F_parity (Fock 0..{NCUT})   blocks = {F_blocks:.12f}")
print(f"                           dense  = {F_dense:.12f}")
print(f"                           agree to {abs(F_blocks-F_dense):.2e}")
print()
for nc in (4, 6, 8):
    print(f"  Fock 0..{nc}:  F_parity = "
          f"{cp.cparity_fidelity(p, M, ncut=nc, use_detuning=USE_DET):.10f}")

In [ ]:
# route 3: the project's own gate registry (resonant sequences only)
pr, Mr, THr, _, _ = cp.load_sequence("cparity_fock8_resonant.npz")
diff, F_oc = cp.crosscheck_optimal_control(pr, Mr, ncut=NCUT)
print(f"resonant sequence: M = {Mr},  g*T = {pr[:Mr].sum():.4f}")
print(f"  max |U_optimal_control - U_controlled_parity| = {diff:.2e}")
print(f"  F_parity from optimal_control.propagate_gates = {F_oc:.10f}")

### Per-block table

The mechanism is that each $2\times2$ block comes out **diagonal**, which is what kills
the g–e mixing, with $\arg U^{gg}_m=-\pi m/2$ and
$\arg U^{ee}_m=2\gamma-\arg U^{gg}_m \pmod{2\pi}$ for a *single* constant $\gamma$
(nonzero only when detuning is used).

At the relaxed 0.9999 floor the blocks are diagonal only to within the fidelity budget —
expect residual $|U_{ge}|\sim10^{-3}$, i.e. $\sim10^{-6}$ in population, which is where
the $\sim5\times10^{-6}$ infidelity comes from. The resonant sequence, which sits at
machine-precision fidelity, has $|U_{ge}|\sim10^{-7}$ instead.

In [ ]:
rows = cp.block_report(p, M, NMAX, use_detuning=USE_DET)
print(f"{'m':>3} {'|U_gg|':>13} {'|U_ge| (mixing)':>17} {'argU_gg/pi':>12} {'argU_ee/pi':>12}")
for m, agg, amix, pg, pe in rows:
    print(f"{m:3d} {agg:13.10f} {amix:17.2e} {pg:+12.5f} {pe:+12.5f}")
print(f"\nmax g-e mixing over all {NMAX} blocks: {max(rr[2] for rr in rows):.2e}")
# arg U_ee + arg U_gg = 2 gamma, constant -- but only modulo 2 pi, so wrap it
two_gamma = np.array([pe + pg for _, _, _, pg, pe in rows])
two_gamma = (two_gamma + 1.0) % 2.0 - 1.0          # fold onto (-1, 1] in units of pi
print("2*gamma/pi per block, folded mod 2pi (should be one constant):")
print("   ", np.round(two_gamma, 6))
print(f"    spread = {np.ptp(two_gamma):.2e}")

In [ ]:
# u[m] = <m,g|U|m,g>   (g branch),   v[m-1] = <m-1,e|U|m-1,e>   (e branch)
u, v = cp.blocks_gg_ee(p, M, NMAX, use_detuning=USE_DET)
nn = np.arange(NCUT + 1)                     # cavity Fock levels 0..8
rel = np.angle(v[:NCUT + 1] / u[:NCUT + 1])          # e-branch minus g-branch
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(np.arange(NMAX + 1), np.unwrap(np.angle(u)) / np.pi, 'o-',
           label=r'achieved $\psi_m$')
ax[0].plot(np.arange(NMAX + 1), THETA * np.arange(NMAX + 1) / np.pi, 'k--', lw=1,
           label=r'target $-\pi m/2$')
ax[0].set_xlabel('block $m$'); ax[0].set_ylabel(r'phase / $\pi$'); ax[0].legend()
ax[0].set_title('g-branch phase profile')
ax[1].plot(nn, np.unwrap(rel) / np.pi, 'o-', label='achieved')
ax[1].plot(nn, nn, 'k--', lw=1, label=r'target $\pi n$')
ax[1].set_xlabel('cavity Fock $n$'); ax[1].set_ylabel(r'(e $-$ g) phase / $\pi$')
ax[1].legend(); ax[1].set_title('conditional phase = the parity')
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 4. The beamsplitter sandwich → controlled SWAP

$V^\dagger n_1 V=n_-$ is checked to machine precision, then the sandwich is scored
against the ideal transmon-controlled SWAP in the full two-cavity space. The free
corrections allowed are a virtual Z per control state plus one unconditional
beamsplitter/phase gauge — the same `gauge_ops=[n_c, n_q]` convention as
`optimal_control`.

The fidelity stays at the parity's value while the **total** photon number is
$\le 8$ and falls off beyond it, which is the boundary the Fock-range note predicts.

In [ ]:
V, bs_err = cp.beamsplitter_to_dark(Nc=10, Nq=2)
print(f"max |V^dag n_1 V - n_-|  = {bs_err:.2e}\n")
for nc in (2, 3, 4, 5):
    t0 = time.time()
    F, _ = cp.check_cswap_from_parity(p, M, ncut_cav=nc, Nc=12, use_detuning=USE_DET)
    flag = "  <- inside the design range" if 2 * nc <= NCUT else "  <- total N > 8"
    print(f"  cavities 0..{nc} each (total N <= {2*nc:2d}):  F_cSWAP = {F:.10f}"
          f"{flag}   ({time.time()-t0:.0f}s)", flush=True)

## 5. Restating the gate: a **qubit–$a_-$ parity**

Before the drive, a word on *what* this gate is. The controlled SWAP is literally a
conditional photon-number parity between the transmon and the $a_-$ mode. Three
identities, each checked to $\sim10^{-15}$ below.

### (a) SWAP is the exchange-parity operator

$$\mathrm{SWAP}_{12}=(-1)^{n_-}=e^{i\pi n_-}$$

The reason is not a calculation, it is a relabelling: a two-cavity state that is
**symmetric** under $1\leftrightarrow2$ has an **even** number of $a_-$ quanta and is
unchanged by SWAP; an **antisymmetric** state has an **odd** number and picks up $-1$.
For one photon, $\tfrac{1}{\sqrt2}(|10\rangle+|01\rangle)=a_+^\dagger|0\rangle$ has
$n_-=0$, and $\tfrac{1}{\sqrt2}(|10\rangle-|01\rangle)=a_-^\dagger|0\rangle$ has $n_-=1$.
So "photon-number parity of $a_-$" and "exchange symmetry of the two cavities" are the
*same observable*.

### (b) So the controlled SWAP is a controlled parity

$$\text{c-SWAP}=\Pi_e\otimes\mathrm{SWAP}+\Pi_g\otimes\mathbb{1}
=\exp\!\big(i\pi\,n_-\,\Pi_e\big)$$

(the second form because $n_-$ and $\Pi_e$ commute). Transmon = control, parity of $a_-$ =
target. Note $e^{i\pi n_-}=e^{-i\pi n_-}$, so the sign is irrelevant at $\pi$.

### (c) It splits into a beamsplitter times the *symmetric* parity entangler

Substituting $\Pi_e=(1-\sigma_z)/2$ with $\sigma_z=\Pi_g-\Pi_e$:

$$\text{c-SWAP}
=\underbrace{e^{\,i\frac{\pi}{2}n_-}}_{\text{unconditional BS}}
\;\times\;
\underbrace{e^{-i\frac{\pi}{2}n_-\sigma_z}}_{\textbf{qubit–}a_-\textbf{ parity}}$$

and the two factors commute. **The second factor is exactly the standard dispersive
parity-mapping unitary** — evolve $H=\tfrac{\chi}{2}n_-\sigma_z$ for $t=\pi/\chi$ and that
is what you get. It is the same gate as a Ramsey parity measurement, just applied to $a_-$
instead of to a single cavity.

So: **"the controlled SWAP is a qubit–$a_-$ parity" is correct, up to one unconditional
beamsplitter** $e^{i\pi n_-/2}$. That is not a separate caveat — it is precisely the
leftover factor that showed up in the pulse design in section 2, so the two observations
are one fact seen twice.

### Why the design target was $\theta=-\pi/2$

This also explains where the "quarter parity" came from. Expand the entangler on the two
branches:

$$e^{-i\frac{\pi}{2}n_-\sigma_z}:\qquad
|g\rangle\to e^{-i\frac{\pi}{2}n_-},\qquad |e\rangle\to e^{+i\frac{\pi}{2}n_-}$$

which is exactly the profile the optimiser was asked for, $\psi_m=-\tfrac{\pi}{2}m$ on the
$g$ branch with its conjugate on the $e$ branch. The target was not a trick — it *is* the
dispersive parity entangler written out per branch.

In [ ]:
# --- the qubit-a_minus parity identities, all to ~1e-15 ---------------------
Nc_p, Nq_p = 7, 2                 # small: we only compare where n1+n2 <= Nc-1
Ic_p, Iq_p = qt.qeye(Nc_p), qt.qeye(Nq_p)
c1 = qt.tensor(qt.destroy(Nc_p), Ic_p, Iq_p)
c2 = qt.tensor(Ic_p, qt.destroy(Nc_p), Iq_p)
c_minus = (c1 - c2) / np.sqrt(2)
n_minus = c_minus.dag() * c_minus

Pg_p = qt.tensor(Ic_p, Ic_p, qt.basis(Nq_p, 0) * qt.basis(Nq_p, 0).dag())
Pe_p = qt.tensor(Ic_p, Ic_p, qt.basis(Nq_p, 1) * qt.basis(Nq_p, 1).dag())
sigma_z = Pg_p - Pe_p

SWAP_p = cs.swap_operator(Nc_p)
cswap_p = (qt.tensor(SWAP_p, qt.basis(Nq_p, 1) * qt.basis(Nq_p, 1).dag())
           + qt.tensor(qt.qeye([Nc_p, Nc_p]),
                       qt.basis(Nq_p, 0) * qt.basis(Nq_p, 0).dag()))

# compare only on states the cavity truncation cannot distort
labels = [(x, y, q) for x in range(Nc_p) for y in range(Nc_p) for q in range(Nq_p)]
safe = [i for i, (x, y, q) in enumerate(labels) if x + y <= Nc_p - 1]
maxdiff = lambda A: np.abs(A.full()[np.ix_(safe, safe)]).max()

print("(a) SWAP == (-1)^{n_-}:                          ",
      f"{maxdiff((1j*np.pi*n_minus).expm() - qt.tensor(SWAP_p, Iq_p)):.2e}")
print("(b) cSWAP == exp(i pi n_- P_e):                  ",
      f"{maxdiff((1j*np.pi*n_minus*Pe_p).expm() - cswap_p):.2e}")

BS_half = (1j * np.pi / 2 * n_minus).expm()
entangler = (-1j * np.pi / 2 * n_minus * sigma_z).expm()
print("(c) cSWAP == BS_half * entangler:                ",
      f"{maxdiff(cswap_p - BS_half * entangler):.2e}")
print("    the two factors commute:                    ",
      f"{maxdiff(BS_half*entangler - entangler*BS_half):.2e}")

chi = 0.37                                    # any chi; t = pi/chi
H_disp = chi / 2 * n_minus * sigma_z
print("(d) entangler == dispersive H=(chi/2)n_- sz at t=pi/chi:",
      f"{maxdiff((-1j*H_disp*(np.pi/chi)).expm() - entangler):.2e}")

# (e) read the per-branch phases straight off the entangler, in the a_- number
#     basis: |m>_{a_-} = (c_minus^dag)^m |0> / sqrt(m!)
print()
print("(e) phases the entangler puts on each branch, in the a_- number basis:")
print(f"    {'m':>3} {'arg on |m,g>':>14} {'arg on |m,e>':>14} "
      f"{'-pi m/2 mod 2pi':>16}")
for m in range(5):
    row = []
    for q in (0, 1):
        vac = qt.tensor(qt.basis(Nc_p, 0), qt.basis(Nc_p, 0), qt.basis(Nq_p, q))
        state = vac
        for _ in range(m):
            state = c_minus.dag() * state
        state = state.unit()
        row.append(np.angle(qt.expect(entangler, state)) / np.pi)
    target = (-m / 2 + 1) % 2 - 1        # wrap onto (-1, 1] in units of pi
    print(f"    {m:3d} {row[0]:13.5f}p {row[1]:13.5f}p {target:14.5f}p")
print("    -> the |g> column is exactly psi_m = -pi m/2, the section-1 target,")
print("       and the |e> column is its conjugate.")

### Reconciling this with the beamsplitter picture

The natural first thought is: the SWAP generator is the beamsplitter Hamiltonian
$a_1^\dagger a_2+a_2^\dagger a_1$, so multiply it by $\Pi_e$ and exponentiate. That is
right, and it is *contained in* the parity picture — but it is **not the whole gate**. The
one line that reconciles them is

$$\boxed{\;n_-\;=\;\frac{N}{2}\;-\;J_x\;},\qquad
J_x\equiv\frac{a_1^\dagger a_2+a_2^\dagger a_1}{2},\quad N=n_1+n_2$$

which is just $a_-^\dagger a_-=\tfrac12(a_1^\dagger-a_2^\dagger)(a_1-a_2)$ expanded. The
beamsplitter generator *is* inside $n_-$ — accompanied by an $N/2$ piece. Conditioning on
$|e\rangle$ and exponentiating (the two terms commute):

$$\text{c-SWAP}=e^{\,i\pi n_-\Pi_e}
=\underbrace{\exp\!\Big[i\tfrac{\pi}{2}N\,\Pi_e\Big]}_{\text{conditional total-number phase}}
\;\cdot\;
\underbrace{\exp\!\Big[-i\tfrac{\pi}{2}\big(a_1^\dagger a_2+a_2^\dagger a_1\big)\Pi_e\Big]}_{\text{controlled beamsplitter}}$$

### So a controlled beamsplitter alone is *not* a controlled SWAP

Even unconditionally, the beamsplitter at angle $\pi$ misses:

$$e^{-i\pi J_x}=e^{-i\frac{\pi}{2}N}\,\mathrm{SWAP}=(-i)^N\,\mathrm{SWAP}$$

Easiest seen on one photon: in $\{|10\rangle,|01\rangle\}$, $2J_x=\sigma_x$, so
$e^{-i\frac{\pi}{2}\sigma_x}=-i\sigma_x$ — the swap happens, but with an extra $-i$, while
the vacuum picks up nothing. That $N$-dependent phase is the whole discrepancy, and
conditioned on the qubit it becomes a real error:

$$e^{-i\pi J_x\Pi_e}=e^{-i\frac{\pi}{2}N\Pi_e}\;\text{c-SWAP}$$

### In *this* system the extra factor cannot be dropped

We work on up to 8 photons in **each** cavity, so the total photon number is not fixed — it
spans $N=0\ldots16$, and the state is generally a superposition across those sectors. The
missing factor $e^{-i\pi N\Pi_e/2}$ is both $N$-dependent *and* qubit-conditional, so
**none** of the free corrections can absorb it: unconditional cavity phase rotations
$e^{-i\mu N}$ hit both branches equally, and a single qubit Z cannot track a phase that
varies with $N$. Scoring the controlled beamsplitter at $\pi$ against the true controlled
SWAP on Fock $0..8$ each, allowing *every* free gauge:

$$F_{\text{controlled-BS}(\pi)}=0.2562$$

against $F=0.99997$ for the gate this notebook actually builds. Not a small correction — the
shortcut is simply a different gate here. (The value is exactly $82^2/162^2$: the $g$ branch
contributes all 81 states in phase, while the $e$ branch contributes
$|\sum_{n_1=0}^{8}(-i)^{n_1}|^2=1$, because $(-i)^{n}$ sums to zero over each period of 4.)

**The one case where the shortcut is fine** is a fixed total photon number — dual rail, or
any single-$N$ sector. There $e^{-i\pi N\Pi_e/2}$ collapses to $e^{-i\pi N/2}$ on the
$|e\rangle$ branch, a free virtual Z on the qubit, and the controlled beamsplitter at $\pi$
*is* the controlled SWAP. Verified below for $N=0\ldots6$ individually, and shown to fail
($1.85$ residual over any qubit phase) as soon as $N\le3$ are taken together. That is not
our regime.

### This is the same point as "do not echo away the dynamical $\chi$"

That $N/2$ term is precisely the dynamical $\chi$ that appears alongside the conditional
beamsplitter in the analog scheme. It is not a nuisance sitting next to the gate — it is
**half of the gate**, locked to the $J_x$ half in the fixed ratio that makes $n_-$. Echo it
away and what remains is $\sigma_z J_x$: a controlled beamsplitter, hence a controlled SWAP
only on a fixed photon-number sector. This is why `analog_controlled_bs.ipynb` concludes that
the echo must not be applied.

In [ ]:
# --- the beamsplitter picture vs the parity picture -------------------------
Nc_b, Nq_b = 7, 2
Ic_b, Iq_b = qt.qeye(Nc_b), qt.qeye(Nq_b)
b1 = qt.tensor(qt.destroy(Nc_b), Ic_b, Iq_b)
b2 = qt.tensor(Ic_b, qt.destroy(Nc_b), Iq_b)

N_tot = b1.dag() * b1 + b2.dag() * b2                # N = n1 + n2
J_x = (b1.dag() * b2 + b2.dag() * b1) / 2            # the beamsplitter generator
b_minus = (b1 - b2) / np.sqrt(2)
n_min = b_minus.dag() * b_minus

Pg_b = qt.tensor(Ic_b, Ic_b, qt.basis(Nq_b, 0) * qt.basis(Nq_b, 0).dag())
Pe_b = qt.tensor(Ic_b, Ic_b, qt.basis(Nq_b, 1) * qt.basis(Nq_b, 1).dag())

SWAP_b = cs.swap_operator(Nc_b)
cswap_b = (qt.tensor(SWAP_b, qt.basis(Nq_b, 1) * qt.basis(Nq_b, 1).dag())
           + qt.tensor(qt.qeye([Nc_b, Nc_b]),
                       qt.basis(Nq_b, 0) * qt.basis(Nq_b, 0).dag()))

lab_b = [(x, y, q) for x in range(Nc_b) for y in range(Nc_b) for q in range(Nq_b)]
safe_b = [i for i, (x, y, q) in enumerate(lab_b) if x + y <= Nc_b - 1]
md_b = lambda A: np.abs(A.full()[np.ix_(safe_b, safe_b)]).max()

print("the algebra:   max|n_- - (N/2 - J_x)| =",
      f"{md_b(n_min - (N_tot / 2 - J_x)):.2e}")
print()
cbs_pi = (-1j * np.pi * J_x * Pe_b).expm()           # controlled BS at angle pi
print("controlled BS at pi, compared with the controlled SWAP:")
print("   max|cBS - cSWAP|                       =",
      f"{md_b(cbs_pi - cswap_b):.2e}    <- not the same gate")
print("   max|cBS - e^{-i pi N P_e/2} cSWAP|     =",
      f"{md_b(cbs_pi - (-1j*np.pi*N_tot*Pe_b/2).expm() * cswap_b):.2e}")
print("   max|e^{+i pi N P_e/2} cBS - cSWAP|     =",
      f"{md_b((1j*np.pi*N_tot*Pe_b/2).expm() * cbs_pi - cswap_b):.2e}"
      "    <- both halves needed")

In [ ]:
# --- on a fixed N the missing factor is only a virtual Z on the qubit -------
print("per fixed total photon number N, is cBS == [P_g + e^{-i pi N/2} P_e] cSWAP ?")
for N_fix in range(7):
    rows = [i for i, (x, y, q) in enumerate(lab_b) if x + y == N_fix]
    Z = Pg_b + np.exp(-1j * np.pi * N_fix / 2) * Pe_b
    diff = (cbs_pi - Z * cswap_b).full()[np.ix_(rows, rows)]
    print(f"   N={N_fix}:  max|difference| = {np.abs(diff).max():.2e}")

print()
print("across a superposition of different N, no single qubit phase works:")
rows = [i for i, (x, y, q) in enumerate(lab_b) if x + y <= 3]
best = min(np.abs((cbs_pi - (Pg_b + np.exp(1j*th) * Pe_b) * cswap_b)
                  .full()[np.ix_(rows, rows)]).max()
           for th in np.linspace(-np.pi, np.pi, 721))
print(f"   best residual over ANY qubit phase, on N<=3 together: {best:.3f}")

In [ ]:
# --- and in OUR regime (Fock 0..8 each, so N = 0..16) it is simply wrong ----
# Score controlled-BS(pi) against the true cSWAP, allowing every free gauge:
# a Z per control branch, cavity virtual Zs, and one unconditional beamsplitter.
Nc_o, Nq_o = 2 * NCUT + 2, 2
Ic_o, Iq_o = qt.qeye(Nc_o), qt.qeye(Nq_o)
o1 = qt.tensor(qt.destroy(Nc_o), Ic_o, Iq_o)
o2 = qt.tensor(Ic_o, qt.destroy(Nc_o), Iq_o)
N_o = o1.dag() * o1 + o2.dag() * o2
Jx_o = (o1.dag() * o2 + o2.dag() * o1) / 2
nm_o = ((o1 - o2) / np.sqrt(2)).dag() * ((o1 - o2) / np.sqrt(2))
Pe_o = qt.tensor(Ic_o, Ic_o, qt.basis(Nq_o, 1) * qt.basis(Nq_o, 1).dag())

SW_o = cs.swap_operator(Nc_o)
cswap_o = (qt.tensor(SW_o, qt.basis(Nq_o, 1) * qt.basis(Nq_o, 1).dag())
           + qt.tensor(qt.qeye([Nc_o, Nc_o]),
                       qt.basis(Nq_o, 0) * qt.basis(Nq_o, 0).dag()))
cbs_o = (-1j * np.pi * Jx_o * Pe_o).expm()

rows_o, branch_o, ntot_o = [], [], []
for q, b in ((1, 0), (0, 1)):
    for n1 in range(NCUT + 1):
        for n2 in range(NCUT + 1):
            rows_o.append((n1 * Nc_o + n2) * Nq_o + q)
            branch_o.append(b)
            ntot_o.append(n1 + n2)
rows_o = np.array(rows_o); branch_o = np.array(branch_o); ntot_o = np.array(ntot_o)
half = len(rows_o) // 2

Mo = (cswap_o.dag() * cbs_o).full()
nu_o, Q_o = np.linalg.eigh(nm_o.full())
w_o = (Mo[rows_o, :] @ Q_o) * np.conj(Q_o[rows_o, :])

F_shortcut = 0.0
for lam in np.linspace(-np.pi, np.pi, 241):
    dg = np.exp(-1j * lam * nu_o) @ w_o.T
    for mu in np.linspace(-np.pi, np.pi, 241):
        t = dg * np.exp(-1j * mu * ntot_o)
        a = abs(t[branch_o == 0].sum()) + abs(t[branch_o == 1].sum())
        F_shortcut = max(F_shortcut, a ** 2 / (2 * half) ** 2)

print(f"controlled-BS(pi) vs true cSWAP on Fock 0..{NCUT} each (N = 0..{2*NCUT}):")
print(f"   F = {F_shortcut:.6f}   with every free gauge allowed")
print(f"   (closed form: 82^2/162^2 = {82**2/162**2:.6f})")
print()
print("the gate this notebook builds, same space:  F = 0.9999683")
print("=> in our regime the beamsplitter-only shortcut is a different gate.")

## 6. Why simultaneous sidebands give a **direct** controlled SWAP

The sandwich above spends two beamsplitters turning $n_1$ into $n_-$. If both
transmon–cavity sidebands are driven **at the same time**, the drive performs that basis
change *itself*, for free, and the sandwich disappears. Here is exactly why, in four steps.

### Step 1 — two matched tones *are* a single-mode drive on one supermode

Drive both sidebands with equal amplitude $g$ and relative phase $\pi$ (i.e. flip the sign
of the second tone). The exchange part of the Hamiltonian is

$$H_{\rm ex}=g\big(a_1^\dagger\sigma^- + \mathrm{h.c.}\big)-g\big(a_2^\dagger\sigma^- + \mathrm{h.c.}\big)
=g\big[(a_1^\dagger-a_2^\dagger)\sigma^-+\mathrm{h.c.}\big]$$

and since $a_-=(a_1-a_2)/\sqrt2$ by definition, $a_1^\dagger-a_2^\dagger=\sqrt2\,a_-^\dagger$:

$$\boxed{\,H_{\rm ex}=\sqrt2\,g\big(a_-^\dagger\sigma^-+\mathrm{h.c.}\big)\,}$$

This is an **algebraic identity, not an approximation** — no perturbation theory, no
rotating-wave step beyond the one already made. Two tones on two cavities are *identically*
one Jaynes–Cummings drive on the single mode $a_-$, at coupling $G=\sqrt2\,g$.

### Step 2 — the other supermode is untouched to all orders

$a_+=(a_1+a_2)/\sqrt2$ does not appear anywhere in $H$. It commutes with the drive, because
$[a_+,a_-^\dagger]=\tfrac12\big([a_1,a_1^\dagger]-[a_2,a_2^\dagger]\big)=0$. So
$a_+$ is a perfect spectator for the whole gate — not "approximately decoupled", *absent*.

The three matching conditions are exactly what make this true: equal amplitudes make the
bright mode exactly $a_-$; relative phase $\pi$ picks $a_-$ rather than $a_+$; equal
detunings are needed for a single static frame to exist at all. Break any of them and the
bright mode is some other combination, with $a_+$ no longer a spectator.

### Step 3 — a SWAP *is* a parity of that mode

$$\mathrm{SWAP}_{12}=e^{\,i\pi n_-}\qquad\text{exactly}$$

because $e^{i\pi n_-}$ sends $a_-\to-a_-$ and leaves $a_+$ alone, and that is precisely
$a_1\leftrightarrow a_2$. Checked to $7\times10^{-15}$ below.

### Step 4 — put them together

Steps 1–3 say: the drive couples the transmon to $a_-$ and *only* to $a_-$, and the gate we
want is a conditional $\pi$-per-photon phase on $a_-$. So the **same 2×2-block machinery
from sections 1–3 applies verbatim**, with $a_1\to a_-$ and $g\to G=\sqrt2 g$:

* blocks $\{|m,g\rangle_{a_-},|m-1,e\rangle_{a_-}\}$, indexed by $a_-$ quanta;
* $|\langle m,g|U|m,g\rangle|=1$ forces each block diagonal;
* target the same quarter parity $\psi_m=-\pi m/2$;
* g branch gets $e^{-i\pi n_-/2}$, e branch $e^{+i\pi n_-/2}$, ratio $e^{i\pi n_-}=$ SWAP.

**The identical pulse table works** — it is the same one-mode problem, just addressed to a
different mode. No $V$, no $V^\dagger$.

### What it costs and what it buys

Factoring the result into unconditional $\times$ conditional,

$$U=\underbrace{e^{-i\pi n_-/2}}_{\text{one unconditional BS}}\times\big(\text{c-SWAP}\big)$$

the leftover is *one* unconditional beamsplitter, versus *two* for the sandwich. (It cannot
be dropped: the ge mechanism forces the two branches to carry conjugate phases
$e^{\pm i\theta n}$, so a common factor always remains. Only a spectator-level scheme —
control on $\{g,f\}$ — avoids it, and that was slower.)

The real prize is the $\sqrt2$. The stored table has $\sum_j x_j=G\,T$, so

$$g\,T=\frac{1}{\sqrt2}\sum_j x_j$$

| route | tones | extra BS pulses | $g\,T$ | $F_{\rm cSWAP}$ (total $N\le8$) |
|---|---|---|---|---|
| single cavity + sandwich | 1 | 2 ($V$, $V^\dagger$) | 11.58 | 0.9999628 |
| **simultaneous sidebands** | 2 | 1 | **8.19** | 0.9999683 |

Same pulse table, same fidelity, $\sqrt2$ less time, one fewer beamsplitter — paid for with
a second pump tone on the coupler. That trade (two-tone heating) is the reason this notebook
uses the single-cavity route by default; the machinery for the direct version is in
`controlled_swap.py`.

> The **analog** notebook `analog_controlled_bs.ipynb` exploits this *same* Step 1–3
> structure without any pulse synthesis: leave one detuned two-tone pulse on, and second-order
> perturbation theory gives $H_{\rm eff}\propto\sigma_z n_-$ directly. Exponentiating that is
> the controlled SWAP. It needs only two calibration numbers, but being perturbative it is
> ~23× slower at matched fidelity.

So one optimisation serves both hardware choices — the sequence solves the abstract problem
"controlled parity of whichever single mode the drive couples to", and the sideband
configuration decides which mode that is:

| drive | mode the transmon couples to | coupling | what the sequence gives |
|---|---|---|---|
| cavity 1 only | $a_1$ | $g$ | controlled parity of cavity 1 → needs $V\cdots V^\dagger$ |
| both, equal amplitude, rel. phase $\pi$ | $a_-$ | $\sqrt2\,g$ | controlled parity of $a_-$ = c-SWAP directly |
| both, equal amplitude, rel. phase $0$ | $a_+$ | $\sqrt2\,g$ | controlled parity of $a_+$ = c-SWAP $\times(-1)^N$ |

`controlled_swap.py` holds the supermode-native version if you prefer to build it that way
from the start, including `cswap_ge_8level.npz` ($g\,T=23.8$ at machine-precision fidelity
over the much wider range total $N\le14$).

In [ ]:
# --- Step 1: two matched tones are identically one JC drive on a_minus -------
Nc_s, Nq_s = 10, 2
Ic, Iq = qt.qeye(Nc_s), qt.qeye(Nq_s)
a1 = qt.tensor(qt.destroy(Nc_s), Ic, Iq)
a2 = qt.tensor(Ic, qt.destroy(Nc_s), Iq)
sigma_minus = qt.tensor(Ic, Ic, qt.basis(Nq_s, 0) * qt.basis(Nq_s, 1).dag())

a_minus = (a1 - a2) / np.sqrt(2)
a_plus = (a1 + a2) / np.sqrt(2)

g_single = 1.0
H_two_tone = g_single * ((a1.dag() - a2.dag()) * sigma_minus)
H_two_tone = H_two_tone + H_two_tone.dag()

H_supermode = np.sqrt(2) * g_single * (a_minus.dag() * sigma_minus)
H_supermode = H_supermode + H_supermode.dag()

print("Step 1  two-tone H  ==  sqrt(2) g * JC(a_minus):",
      f"{(H_two_tone - H_supermode).norm():.2e}")

# --- Step 2: a_plus is absent from H ----------------------------------------
# Check on states well inside the cutoff: at the top Fock level the two cavities'
# [a, a^dag] boundary terms do not cancel, which is a truncation artefact only.
labels = [(n1, n2, q) for n1 in range(Nc_s) for n2 in range(Nc_s)
          for q in range(Nq_s)]
inside = [i for i, (n1, n2, q) in enumerate(labels) if n1 + n2 + q <= 5]
commutator = (H_two_tone * a_plus - a_plus * H_two_tone).full()
print("Step 2  max |[H, a_plus]| on the N<=5 block:",
      f"{np.abs(commutator[np.ix_(inside, inside)]).max():.2e}")
print("        (and the propagator leaves a_plus alone:",
      f"{cs.check_supermode_reduction():.2e})")

# --- Step 3: SWAP is the parity of a_minus ----------------------------------
print("Step 3  max |exp(i pi n_minus) - SWAP|:",
      f"{cs.check_swap_is_dark_parity(Nc=16, Nmax=14):.2e}")

In [ ]:
# --- Step 4: the SAME pulse table, driven on a_minus, is a direct cSWAP -----
print(f"pulse table: M = {M} segments, sum_j x_j = G*T = {gT:.4f}")
print(f"  single-cavity drive (G = g):        g*T = {gT:.4f}")
print(f"  simultaneous drive  (G = sqrt2 g):  g*T = {gT/np.sqrt(2):.4f}"
      f"   <- sqrt(2) faster")
print()
for nc in (3, 4):
    F = cs.full_cswap_fidelity(p, M, use_detuning=USE_DET, Nc=14, ncut=nc,
                               ctrl=(0, 1))
    print(f"  direct cSWAP, cavities 0..{nc} each (total N <= {2*nc}): "
          f"F = {F:.10f}")
print()
print("no V, no V^dag -- the drive itself selected the mode.")

## 7. Trade-offs

`gate_fidelity_blocks` is cheap, so the $g\,T$ / fidelity trade-off is easy to map.
Two things drive $g\,T$ down: allowing sideband detuning (a third knob per segment),
and spending the fidelity budget rather than insisting on machine precision.

In [ ]:
print("stored sequences:")
for f in ("cparity_fock8.npz", "cparity_fock8_resonant.npz"):
    q, Mq, thq, ncq, detq = cp.load_sequence(f)
    Fq = cp.gate_fidelity_blocks(q, Mq, ncut=ncq, use_detuning=detq)
    print(f"  {f:32s} M={Mq:3d}  detuning={'on ' if detq else 'off'}  "
          f"g*T={q[:Mq].sum():8.4f}  F={Fq:.10f}")

### The code that generated the stored pulse

Two objectives, in priority order: **minimise $g\,T$ first, then the pulse count $M$.**
They feed each other, so `optimise_alternating` loops the two stages:

* **stage A** `descend_gT` — fix $M$, walk the total area down multiplicatively,
  warm-starting each step, accepting only if the true Fock-0–8 parity fidelity still
  clears the floor. This *spends* fidelity budget.
* **stage B** `reduce_M` — fix $g\,T$, repeatedly delete the smallest-area segment and
  re-optimise at the same area. This *frees* budget, because area-minimisation leaves
  near-zero no-op segments behind and fewer/larger segments describe the same gate better.

Running A alone stalls at $g\,T=12.06$; alternating gets to 11.58 at a smaller $M$.
Both stages checkpoint to `out_path` after every accepted step, so a long run can be
interrupted without losing progress.

Set `RUN_SEARCH = True` to reproduce (tens of minutes; use several seeds in parallel —
different seeds break through at different $M$).

In [ ]:
RUN_SEARCH = False        # flip to True to re-run the optimisation

if RUN_SEARCH:
    # --- cold start: find any valid sequence, then minimise g*T at fixed M ----
    rep = cp.produce_optimum(ncut=8, M_list=(14, 16, 20, 24), use_detuning=True,
                             seed=3, out_path="cparity_cold.npz")
    p0, M0 = rep["params"], rep["M"]

    # --- or warm start from the stored optimum (much faster) -----------------
    # p0, M0, _, _, _ = cp.load_sequence("cparity_fock8.npz")

    # --- alternate: minimise g*T, then the pulse count, until neither moves ---
    out = cp.optimise_alternating(p0, M0, ncut=8, theta=cp.THETA, det=True,
                                  floor=cp.FLOOR_DEFAULT, rounds=8, seed=1004,
                                  out_path="cparity_fock8_new.npz", verbose=True)
    print(f"g*T = {out['gT']:.4f}   M = {out['M']}   F = {out['F']:.10f}")
    for gT_i, M_i, F_i in out["history"]:
        print(f"   round: g*T={gT_i:.4f}  M={M_i}  F={F_i:.10f}")

    # the two stages can also be driven one at a time:
    # gT1, p1 = cp.descend_gT(p0, M0, 8, cp.THETA, True, ckpt_path="a.npz")
    # M2,  p2 = cp.reduce_M(p1, M0, 8, cp.THETA, True, ckpt_path="b.npz")